# Credit Card Fraud Detection — Preprocessing & Feature Engineering

Prepare the dataset for model development while keeping the train/test split and training-derived transformations leakage-safe.

## 1. Imports and Configuration

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.features import add_time_features, add_training_volume_feature
from src.preprocessing import make_preprocessor

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Class"


## 2. Load the Dataset

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset path: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape}")


Dataset path: D:\3\JupyterProjects\ML Projects\fraud-detection\data\creditcard.csv
Shape: (284807, 31)


## 3. Separate Features and Target

`Class` is the binary target (`0` = legitimate, `1` = fraud). It is removed before feature engineering and preprocessing.

In [3]:
TARGET = "Class"

X = df.drop(columns=TARGET).copy()
y = df[TARGET].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Fraud rate: {y.mean():.4%}")


Feature matrix shape: (284807, 30)
Target shape: (284807,)
Fraud rate: 0.1727%


## 4. Stratified Train/Test Split

Split before fitting any transformation. Stratification preserves the fraud rate across the two subsets.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training rows:   {len(X_train):,}")
print(f"Testing rows:    {len(X_test):,}")
print(f"Training fraud rate: {y_train.mean():.4%}")
print(f"Testing fraud rate:  {y_test.mean():.4%}")


Training rows:   227,845
Testing rows:    56,962
Training fraud rate: 0.1729%
Testing fraud rate:  0.1720%


## 5. Feature Engineering

`Time` is converted into elapsed hours and cyclical hour-of-day features. Sine/cosine encoding preserves the adjacency between hour 23 and hour 0.

In [5]:
X_train_fe = add_time_features(X_train)
X_test_fe = add_time_features(X_test)

print("Engineered training shape:", X_train_fe.shape)
print("Engineered test shape:", X_test_fe.shape)

Engineered training shape: (227845, 34)
Engineered test shape: (56962, 34)


In [6]:
X_train_fe = add_time_features(X_train)
X_test_fe = add_time_features(X_test)

display(X_train_fe[["Time", "Time_Hours", "Hour", "Hour_Sin", "Hour_Cos"]].head())


,Time,Time_Hours,Hour,Hour_Sin,Hour_Cos
265518,161919.0,44.977500,20,-8.660254e-01,0.500000
180305,124477.0,34.576944,10,5.000000e-01,-0.866025
42664,41191.0,11.441944,11,2.588190e-01,-0.965926
198723,132624.0,36.840000,12,1.224647e-16,-1.000000
82325,59359.0,16.488611,16,-8.660254e-01,-0.500000


## 6. Transaction Volume Indicator

`Transactions_Per_Hour` measures activity in each elapsed-time hour. The hourly counts are learned from the training set and mapped to the test set; unseen test buckets receive zero.

In [7]:
X_train_fe, X_test_fe, hourly_counts = add_training_volume_feature(
    X_train_fe,
    X_test_fe,
)

print("Hourly buckets learned:", len(hourly_counts))
print("Training shape:", X_train_fe.shape)
print("Test shape:", X_test_fe.shape)

Hourly buckets learned: 48
Training shape: (227845, 35)
Test shape: (56962, 35)


In [8]:
X_train_fe, X_test_fe, hourly_counts = add_training_volume_feature(
    X_train_fe,
    X_test_fe,
)

display(
    X_train_fe[
        ["Time", "Hour", "Transactions_Per_Hour"]
    ].head()
)

print(f"Unique hourly buckets learned from training data: {len(hourly_counts)}")


,Time,Hour,Transactions_Per_Hour
265518,161919.0,20,6266.0
180305,124477.0,10,6597.0
42664,41191.0,11,6825.0
198723,132624.0,12,6192.0
82325,59359.0,16,6200.0


Unique hourly buckets learned from training data: 48


## 7. Check the Engineered Features

In [9]:
assert TARGET not in X_train_fe.columns
assert TARGET not in X_test_fe.columns
assert list(X_train_fe.columns) == list(X_test_fe.columns)

print("Train/test feature schemas match.")
print(f"Number of features before preprocessing: {X_train_fe.shape[1]}")


Train/test feature schemas match.
Number of features before preprocessing: 35


## 8. Define the Preprocessing Pipeline

`Time` and `Amount` are scaled with `RobustScaler`, which uses the median and interquartile range and is less sensitive to extreme values. The PCA features and engineered features pass through unchanged.

In [10]:
preprocessing_pipeline = make_preprocessor()

print(preprocessing_pipeline)

ColumnTransformer(remainder='passthrough',
                  transformers=[('robust_scaler', RobustScaler(),
                                 ['Time', 'Amount'])],
                  verbose_feature_names_out=False)


## 9. Fit on Training Data Only

The preprocessor is fitted on `X_train_fe` and then applied to the test set.

In [11]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train_fe)
X_test_processed = preprocessing_pipeline.transform(X_test_fe)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed testing shape:  {X_test_processed.shape}")


Processed training shape: (227845, 35)
Processed testing shape:  (56962, 35)


## 10. Recover Processed Feature Names

Retain the transformed feature names for model diagnostics and interpretability.

In [12]:
feature_names = preprocessing_pipeline.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_fe.index,
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test_fe.index,
)

display(X_train_processed.head())


,Time,Amount,V1,V2,V3,V4,V5,V6,V7,V8,...,V24,V25,V26,V27,V28,Time_Hours,Hour,Hour_Sin,Hour_Cos,Transactions_Per_Hour
265518,0.905774,-0.204315,1.946747,-0.752526,-1.355130,-0.661630,1.502822,4.024933,-1.479661,1.139880,...,0.690980,-0.350316,-0.388907,0.077641,-0.032248,44.977500,20.0,-8.660254e-01,0.500000,6266.0
180305,0.465984,-0.264579,2.035149,-0.048880,-3.058693,0.247945,2.943487,3.298697,-0.002192,0.674782,...,0.707090,0.512885,-0.471198,0.002520,-0.069002,34.576944,10.0,5.000000e-01,-0.866025,6597.0
42664,-0.512286,2.130828,-0.991920,0.603193,0.711976,-0.992425,-0.825838,1.956261,-2.212603,-5.037523,...,-0.932803,0.826684,0.913773,0.038049,0.185340,11.441944,11.0,2.588190e-01,-0.965926,6825.0
198723,0.561678,-0.221294,2.285718,-1.500239,-0.747565,-1.668119,-1.394143,-0.350339,-1.427984,0.010010,...,-0.538236,-0.278032,-0.162068,0.018045,-0.063005,36.840000,12.0,1.224647e-16,-1.000000,6192.0
82325,-0.298886,0.892136,-0.448747,-1.011440,0.115903,-3.454854,0.715771,-0.147490,0.504347,-0.113817,...,-1.362383,-0.292234,-0.144622,-0.032580,-0.064194,16.488611,16.0,-8.660254e-01,-0.500000,6200.0


## 11. Verify Scaling

Training medians for the robust-scaled features should be close to zero. Test-set statistics are not expected to match them exactly.

In [13]:
scaled_features = ["Time", "Amount"]

scaling_check = pd.DataFrame({
    "train_median": X_train_processed[scaled_features].median(),
    "train_IQR": (
        X_train_processed[scaled_features].quantile(0.75)
        - X_train_processed[scaled_features].quantile(0.25)
    ),
    "test_median": X_test_processed[scaled_features].median(),
})

display(scaling_check)


,train_median,train_IQR,test_median
Time,0.0,1.0,-0.003859
Amount,0.0,1.0,0.000000


## 12. Check for Missing or Infinite Values

In [14]:
def validate_numeric_matrix(data, name):
    missing = int(data.isna().sum().sum())
    infinite = int(np.isinf(data.to_numpy()).sum())

    print(f"{name}:")
    print(f"  Missing values:  {missing:,}")
    print(f"  Infinite values: {infinite:,}")

    assert missing == 0
    assert infinite == 0

validate_numeric_matrix(X_train_processed, "Processed training data")
validate_numeric_matrix(X_test_processed, "Processed testing data")


Processed training data:
  Missing values:  0
  Infinite values: 0
Processed testing data:
  Missing values:  0
  Infinite values: 0


## 13. Final Dataset Shapes

The resulting training and test matrices are ready for model development.

In [15]:
print("Final modeling datasets")
print("-" * 35)
print(f"X_train: {X_train_processed.shape}")
print(f"X_test:  {X_test_processed.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")
print()
print(f"Training fraud cases: {int(y_train.sum()):,}")
print(f"Testing fraud cases:  {int(y_test.sum()):,}")


Final modeling datasets
-----------------------------------
X_train: (227845, 35)
X_test:  (56962, 35)
y_train: (227845,)
y_test:  (56962,)

Training fraud cases: 394
Testing fraud cases:  98


## 14. Leakage Checklist

- Target separated before preprocessing
- Stratified train/test split performed first
- `RobustScaler` fitted only on training data
- Test data transformed with training-fitted statistics
- Transaction-volume mapping learned from training data
- No resampling or model fitting in this notebook

**Next:** model comparison with cross-validation and imbalance handling.